<a href="https://colab.research.google.com/github/kady05naik/PySpark/blob/main/dataFrameTransformations/transformations_based_on_dataframe_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformations on Data Frame 11

**UNION() & UNIONBYNAME(), ALLOWMISSINGCOLUMNS()**

In [50]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [4]:
spark=SparkSession.builder \
  .appName('MyApp') \
  .getOrCreate()

In [2]:
data= [
        (1,'Joe',70000,3),
        (2,'Henry',80000,4),
		    (3,'Sam',60000,None),
        (4,'Max',90000,None)
      ]

In [3]:
schema=StructType([
    StructField('id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('salary', LongType(), True),
    StructField('mgr',IntegerType(), True)
])

In [6]:
df=spark.createDataFrame(data,schema)

### **1. Difference Between union() and unionAll() in PySpark**

**Important Rule for Union**

Both DataFrames must have:

1. **Same number of columns**

2. **Same data types**

3. **Same column order**





In [8]:
df.show()

+---+-----+------+----+
| id| name|salary| mgr|
+---+-----+------+----+
|  1|  Joe| 70000|   3|
|  2|Henry| 80000|   4|
|  3|  Sam| 60000|NULL|
|  4|  Max| 90000|NULL|
+---+-----+------+----+



In [10]:
df.union(df) \
  .show()

+---+-----+------+----+
| id| name|salary| mgr|
+---+-----+------+----+
|  1|  Joe| 70000|   3|
|  2|Henry| 80000|   4|
|  3|  Sam| 60000|NULL|
|  4|  Max| 90000|NULL|
|  1|  Joe| 70000|   3|
|  2|Henry| 80000|   4|
|  3|  Sam| 60000|NULL|
|  4|  Max| 90000|NULL|
+---+-----+------+----+



In [12]:
df.unionAll(df) \
  .show()

+---+-----+------+----+
| id| name|salary| mgr|
+---+-----+------+----+
|  1|  Joe| 70000|   3|
|  2|Henry| 80000|   4|
|  3|  Sam| 60000|NULL|
|  4|  Max| 90000|NULL|
|  1|  Joe| 70000|   3|
|  2|Henry| 80000|   4|
|  3|  Sam| 60000|NULL|
|  4|  Max| 90000|NULL|
+---+-----+------+----+



###In PySpark DataFrames, **union() and unionAll()** **behave the same**.

Historically they were **different in SQL**, **but in PySpark unionAll() is deprecated** and union() **is used instead**.



---



### **2. Get Unique Records After Union**

In [13]:
df.union(df) \
  .distinct() \
  .show()

+---+-----+------+----+
| id| name|salary| mgr|
+---+-----+------+----+
|  2|Henry| 80000|   4|
|  1|  Joe| 70000|   3|
|  4|  Max| 90000|NULL|
|  3|  Sam| 60000|NULL|
+---+-----+------+----+





---



### **3. Difference Between union() and unionByName()**

**unionByName()** merges two DataFrames **based on column names** instead of column positions.



```
+-------------------------------+-------------------+----------------------+
|Features               		|		union()		|		unionByName()  |
|-------------------------------|-------------------|----------------------|
|Column matching      			|	By position		|	By column name     |
|Column order required			|	Yes				|	No                 |
|Safe when column order differs	|	No				|	Yes                |
|Duplicate rows					|	Allowed			|	Allowed            |
+-------------------------------+-------------------+----------------------+                                                                          
```

In [21]:
data1= [
        (1,'Joe',70000,3),
        (2,'Henry',80000,4),
		    (3,'Sam',60000,None),
        (4,'Max',90000,None)
      ]

In [22]:
schema1=StructType([
    StructField('id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('salary', LongType(), True),
    StructField('mgr',IntegerType(), True)
])

In [23]:
data2= [
        (1,'Joe',3,70000),
        (2,'Henry',4,80000),
		    (3,'Sam',None,60000),
        (4,'Max',None,90000)
      ]

In [24]:
schema2=StructType([
    StructField('id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('mgr',IntegerType(), True),
    StructField('salary', LongType(), True)
])

In [28]:
df1=spark.createDataFrame(data1,schema1)
df2=spark.createDataFrame(data2, schema2)

In [29]:
df1.unionByName(df2) \
  .show()

+---+-----+------+----+
| id| name|salary| mgr|
+---+-----+------+----+
|  1|  Joe| 70000|   3|
|  2|Henry| 80000|   4|
|  3|  Sam| 60000|NULL|
|  4|  Max| 90000|NULL|
|  1|  Joe| 70000|   3|
|  2|Henry| 80000|   4|
|  3|  Sam| 60000|NULL|
|  4|  Max| 90000|NULL|
+---+-----+------+----+



**unionByName() does allow duplicate**s. It does NOT remove duplicates.



---



In [36]:
df1.union(df2) \
  .show()

+---+-----+------+-----+
| id| name|salary|  mgr|
+---+-----+------+-----+
|  1|  Joe| 70000|    3|
|  2|Henry| 80000|    4|
|  3|  Sam| 60000| NULL|
|  4|  Max| 90000| NULL|
|  1|  Joe|     3|70000|
|  2|Henry|     4|80000|
|  3|  Sam|  NULL|60000|
|  4|  Max|  NULL|90000|
+---+-----+------+-----+



**union() is not working when column order differs**



---



### **4. If Two DataFrames Have Different Columns**
1.   We use **allowMissingColumns=True**
2.   Spark fills missing columns with **NULL**.

In [37]:
data1= [
        (1,'Joe','Liver',70000,3),
        (2,'Henry','Alle',80000,4),
		    (3,'Sam','Smith',60000,None),
        (4,'Max','Roy',90000,None)
      ]

In [44]:
schema1=StructType([
    StructField('id', IntegerType(), True),
    StructField('fname', StringType(), True),
    StructField('lname', StringType(), True),
    StructField('salary', LongType(), True),
    StructField('mgr',IntegerType(), True)
])

In [45]:
data2= [
        (1,'Joe',3,70000),
        (2,'Henry',4,80000),
		    (3,'Sam',None,60000),
        (4,'Max',None,90000)
      ]

In [46]:
schema2=StructType([
    StructField('id', IntegerType(), True),
    StructField('fname', StringType(), True),
    StructField('mgr',IntegerType(), True),
    StructField('salary', LongType(), True)
])

In [47]:
df1=spark.createDataFrame(data1,schema1)
df2=spark.createDataFrame(data2, schema2)

In [48]:
df1.unionByName(df2) \
  .show()

AnalysisException: Cannot resolve column name "lname" among (id, fname, mgr, salary).

In [49]:
df1.unionByName(df2, allowMissingColumns=True) \
  .show()

+---+-----+-----+------+----+
| id|fname|lname|salary| mgr|
+---+-----+-----+------+----+
|  1|  Joe|Liver| 70000|   3|
|  2|Henry| Alle| 80000|   4|
|  3|  Sam|Smith| 60000|NULL|
|  4|  Max|  Roy| 90000|NULL|
|  1|  Joe| NULL| 70000|   3|
|  2|Henry| NULL| 80000|   4|
|  3|  Sam| NULL| 60000|NULL|
|  4|  Max| NULL| 90000|NULL|
+---+-----+-----+------+----+



In [51]:
spark.stop()